# FDA Drug Label RAG — AWS Bedrock (Bearer Token)
### Section 1 — Install & Import

In [1]:
# !pip install boto3 chromadb -q

import json
import os
import time
import boto3
import chromadb

print("Done.")

Done.


---
### Section 2 — Set your Bedrock Bearer Token

In [ ]:
os.environ['AWS_BEARER_TOKEN_BEDROCK'] = ""

client = boto3.client("bedrock-runtime", region_name="ap-southeast-2")

response = client.converse(
    modelId="amazon.nova-pro-v1:0",
    messages=[
        {
            "role": "user",
            "content": [{"text": "Say hello."}]
        }
    ]
)

print(response["output"]["message"]["content"][0]["text"])

Hello! It's nice to meet you. How can I assist you today? Whether you have a question, need information, or just want to chat, feel free to let me know.


---
### Section 3 — Load your chunks JSON

In [3]:
JSON_FILE = "all_drugs_chunks.json"  # change if needed

with open(JSON_FILE) as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")
print()
print("Sample chunk:")
print(json.dumps(chunks[0], indent=2))

Loaded 507 chunks

Sample chunk:
{
  "drug": "levetiracetam",
  "section": "Indications and Usage",
  "subsection": "1.1 Partial-Onset Seizures",
  "title": "1.1 Partial-Onset Seizures",
  "text": "Levetiracetam is indicated for the treatment of partial-onset seizures in patients 1 month of age and older.",
  "loinc_code": "34067-9"
}


---
### Section 4 — Format chunks for embedding

In [4]:
def format_for_embedding(chunk):
    """Prepend metadata so the embedding captures drug + section context."""
    lines = [f"Drug: {chunk['drug']}"]
    lines.append(f"Section: {chunk['section']}")
    if chunk.get('subsection'):
        lines.append(f"Subsection: {chunk['subsection']}")
    lines.append(f"\n{chunk['text']}")
    return '\n'.join(lines)

# Preview first 3
for i, c in enumerate(chunks[:3]):
    print(f"{'='*60}")
    print(f"[Chunk {i}]")
    print(format_for_embedding(c)[:300])
    print()

[Chunk 0]
Drug: levetiracetam
Section: Indications and Usage
Subsection: 1.1 Partial-Onset Seizures

Levetiracetam is indicated for the treatment of partial-onset seizures in patients 1 month of age and older.

[Chunk 1]
Drug: levetiracetam
Section: Indications and Usage
Subsection: 1.2 Myoclonic Seizures in Patients with Juvenile Myoclonic Epilepsy

Levetiracetam is indicated as adjunctive therapy for the treatment of myoclonic seizures in patients 12 years of age and older with juvenile myoclonic epilepsy.

[Chunk 2]
Drug: levetiracetam
Section: Indications and Usage
Subsection: 1.3 Primary Generalized Tonic-Clonic Seizures

Levetiracetam is indicated as adjunctive therapy for the treatment of primary generalized tonic-clonic seizures in patients 6 years of age and older with idiopathic generalized epilepsy.



---
### Section 5 — Embed chunks using Amazon Titan Embed v2

We use a separate bedrock-runtime client for embeddings.
Model: `amazon.titan-embed-text-v2:0`
Cost: ~$0.00002 per 1K tokens — embedding all chunks costs less than $0.001

In [5]:
EMBED_MODEL = "amazon.titan-embed-text-v2:0"
MAX_CHARS = 49000  # safe buffer under Titan's 50k limit

# Separate client for embeddings
embed_client = boto3.client("bedrock-runtime", region_name="ap-southeast-2")

def embed_text(text):
    """Embed a string using Amazon Titan Embed v2. Truncates if over limit."""
    if len(text) > MAX_CHARS:
        print(f"  Warning: chunk truncated from {len(text)} to {MAX_CHARS} chars")
        text = text[:MAX_CHARS]
    
    body = json.dumps({
        "inputText": text,
        "dimensions": 256,
        "normalize": True
    })
    response = embed_client.invoke_model(
        modelId=EMBED_MODEL,
        body=body,
        contentType="application/json",
        accept="application/json"
    )
    result = json.loads(response["body"].read())
    return result["embedding"]

# Quick test
test_emb = embed_text("test")
print(f"Embedding model working.")
print(f"Dimensions: {len(test_emb)}")

# Show any oversized chunks before embedding
print("\nChecking chunk sizes...")
oversized = [(i, chunk) for i, chunk in enumerate(chunks) if len(format_for_embedding(chunk)) > MAX_CHARS]
if oversized:
    print(f"  {len(oversized)} oversized chunks found — will be truncated:")
    for i, chunk in oversized:
        print(f"    Chunk {i}: {chunk['drug']} / {chunk['section']} / {chunk['subsection']} ({len(format_for_embedding(chunk))} chars)")
else:
    print("  All chunks within limit.")

# Embed all chunks
print("\nEmbedding all chunks...")
embeddings = []

for i, chunk in enumerate(chunks):
    text = format_for_embedding(chunk)
    emb  = embed_text(text)
    embeddings.append(emb)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(chunks)} done")
    time.sleep(0.1)

print(f"\nAll {len(embeddings)} chunks embedded.")
print(f"Embedding dimensions: {len(embeddings[0])}")

Embedding model working.
Dimensions: 256

Checking chunk sizes...
  All chunks within limit.

Embedding all chunks...
  10/507 done
  20/507 done
  30/507 done
  40/507 done
  50/507 done
  60/507 done
  70/507 done
  80/507 done
  90/507 done
  100/507 done
  110/507 done
  120/507 done
  130/507 done
  140/507 done
  150/507 done
  160/507 done
  170/507 done
  180/507 done
  190/507 done
  200/507 done
  210/507 done
  220/507 done
  230/507 done
  240/507 done
  250/507 done
  260/507 done
  270/507 done
  280/507 done
  290/507 done
  300/507 done
  310/507 done
  320/507 done
  330/507 done
  340/507 done
  350/507 done
  360/507 done
  370/507 done
  380/507 done
  390/507 done
  400/507 done
  410/507 done
  420/507 done
  430/507 done
  440/507 done
  450/507 done
  460/507 done
  470/507 done
  480/507 done
  490/507 done
  500/507 done

All 507 chunks embedded.
Embedding dimensions: 256


---
### Section 6 — Set up ChromaDB locally

In [6]:
chroma_client = chromadb.PersistentClient(path="./chroma_db_bedrock")

# Clean start
try:
    chroma_client.delete_collection("fda_labels_bedrock")
    print("Deleted existing collection.")
except:
    pass

collection = chroma_client.create_collection(
    name="fda_labels_bedrock",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created.")
print("Storage: ./chroma_db_bedrock")

Deleted existing collection.
ChromaDB collection created.
Storage: ./chroma_db_bedrock


---
### Section 7 — Load chunks into ChromaDB

In [7]:
metadatas = []
for chunk in chunks:
    metadatas.append({
        "drug"      : chunk.get("drug", ""),
        "section"   : chunk.get("section", ""),
        "subsection": chunk.get("subsection") or "",
        "title"     : chunk.get("title", ""),
        "loinc_code": chunk.get("loinc_code", ""),
    })

collection.add(
    ids        = [f"chunk_{i}" for i in range(len(chunks))],
    embeddings = embeddings,
    documents  = [format_for_embedding(c) for c in chunks],
    metadatas  = metadatas
)

print(f"Loaded {collection.count()} chunks into ChromaDB.")

Loaded 507 chunks into ChromaDB.


---
### Section 8 — Test retrieval

In [8]:
def retrieve(question, n_results=5):
    """Embed the question and retrieve the most relevant chunks."""
    query_embedding = embed_text(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    return results

# Test
question = "What is the dose adjustment for renal impairment?"
results  = retrieve(question)

print(f"Question: {question}")
print("=" * 60)
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"\n[Result {i+1}] relevance score: {1 - dist:.3f}")
    print(f"  Drug      : {meta['drug']}")
    print(f"  Section   : {meta['section']}")
    print(f"  Subsection: {meta['subsection']}")
    print(f"  Preview   : {doc[:200]}...")

Question: What is the dose adjustment for renal impairment?

[Result 1] relevance score: 0.626
  Drug      : perampanel
  Section   : Use in Specific Populations
  Subsection: 8.7 Renal Impairment
  Preview   : Drug: perampanel
Section: Use in Specific Populations
Subsection: 8.7 Renal Impairment

Dose adjustment is not required in patients with mild renal impairment. Perampanel should be used with caution i...

[Result 2] relevance score: 0.610
  Drug      : oxcarbazepine
  Section   : Use in Specific Populations
  Subsection: 8.7 Renal Impairment
  Preview   : Drug: oxcarbazepine
Section: Use in Specific Populations
Subsection: 8.7 Renal Impairment

Dose adjustment is recommended for renally impaired patients (creatinine clearance <30 mL/min) [ see Dosage a...

[Result 3] relevance score: 0.567
  Drug      : perampanel
  Section   : Clinical Pharmacology
  Subsection: 
  Preview   : Drug: perampanel
Section: Clinical Pharmacology

Renal Impairment A dedicated study has not been condu

---
### Section 9 — Ask Claude via Bedrock converse API

Using `client.converse()` — the same pattern from your setup code.
Model: `us.anthropic.claude-sonnet-4-6`

In [9]:
LLM_MODEL = "amazon.nova-pro-v1:0"

SYSTEM_PROMPT = """You are a pharmaceutical regulatory assistant at UCB.
Answer questions using ONLY the provided FDA label sections.
Do not use any external knowledge or make anything up.
Always end your answer with a citation in this exact format:
Source: [drug name] — [section name]"""

def build_prompt(question, results):
    """Build the RAG prompt with retrieved context."""
    context_blocks = []
    for i, (doc, meta) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0]
    )):
        block = (
            f"[Source {i+1}]\n"
            f"Drug: {meta['drug']}\n"
            f"Section: {meta['section']}\n"
            f"Subsection: {meta['subsection']}\n"
            f"Text: {doc}\n"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)

    return f"""--- FDA LABEL SECTIONS ---
{context}

--- QUESTION ---
{question}"""


def ask(question):
    """Full RAG pipeline: retrieve → build prompt → call Nova → return answer."""
    results = retrieve(question)
    prompt  = build_prompt(question, results)

    response = client.converse(
        modelId=LLM_MODEL,
        system=[{"text": SYSTEM_PROMPT}],  # Nova supports system prompt same way
        messages=[
            {
                "role": "user",
                "content": [{"text": prompt}]
            }
        ],
        inferenceConfig={
            "maxTokens": 1024,
            "temperature": 0
        }
    )

    return response["output"]["message"]["content"][0]["text"]


# Test it
answer = ask("What is the dose adjustment for renal impairment in levetiracetam?")
print(answer)

The dose adjustment for renal impairment in levetiracetam is based on the patient's creatinine clearance (CLcr) adjusted for body surface area (BSA). The recommended dosage adjustments for adults are as follows:

- Normal renal function (CLcr > 80 mL/min/1.73m²): 500 to 1,500 mg every 12 hours.
- Mild renal impairment (CLcr 50 – 80 mL/min/1.73m²): 500 to 1,000 mg every 12 hours.
- Moderate renal impairment (CLcr 30 – 50 mL/min/1.73m²): 250 to 750 mg every 12 hours.
- Severe renal impairment (CLcr < 30 mL/min/1.73m²): 250 to 500 mg every 12 hours.
- End-stage renal disease (ESRD) patients using dialysis: 500 to 1,000 mg every 24 hours. A supplemental dose of 250 to 500 mg is recommended following dialysis.

To calculate the dose, first estimate the patient’s creatinine clearance (CLcr) using the formula: 

CLcr= [140-age (years)] x weight (kg) (x 0.85 for female patients) / (72 x serum creatinine (mg/dL))

Then adjust CLcr for BSA:

CLcr (mL/min/1.73 m²) = CLcr (mL/min) x 1.73 / BSA sub

### More Question and Performance Metrics

In [10]:
import time
import statistics

questions = [
    # Levetiracetam — dosing
    "What is the starting dose of levetiracetam for adults?",
    "What is the maximum daily dose of levetiracetam?",
    "What is the recommended dose of levetiracetam for a 5-year-old child?",
    "How should levetiracetam be dosed in patients on dialysis?",
    "What is the dose adjustment for levetiracetam in moderate renal impairment?",

    # Levetiracetam — safety & pharmacology
    "Can levetiracetam cause aggression in children?",
    "What dermatological reactions have been reported for levetiracetam?",
    "What is the mechanism of action of levetiracetam?",
    "What is the half-life of levetiracetam in adults?",
    "Is levetiracetam excreted in breast milk?",

    # Brivaracetam — dosing
    "What is the starting dose of brivaracetam for adults?",
    "What is the maximum daily dose of brivaracetam?",
    "How should brivaracetam be dosed in patients with hepatic impairment?",
    "Is dose adjustment required for brivaracetam in renal impairment?",
    "Can brivaracetam be given intravenously?",

    # Brivaracetam — safety & pharmacology
    "What are the most common adverse reactions to brivaracetam?",
    "Does brivaracetam affect the QT interval?",
    "What is the half-life of brivaracetam?",
    "Does alcohol interact with brivaracetam?",
    "Is brivaracetam a controlled substance?",

    # Lacosamide
    "What is the mechanism of action of lacosamide?",
    "What is the recommended starting dose of lacosamide for adults?",
    "Can lacosamide be given intravenously?",
    "What cardiac effects are associated with lacosamide?",
    "Is lacosamide approved for pediatric patients?",

    # Lamotrigine
    "What is the titration schedule for lamotrigine?",
    "What serious skin reactions are associated with lamotrigine?",
    "How does valproate affect lamotrigine dosing?",
    "Is lamotrigine approved for bipolar disorder?",
    "What are the contraindications for lamotrigine?",

    # Topiramate
    "What are the most common side effects of topiramate?",
    "Can topiramate cause kidney stones?",
    "What is the mechanism of action of topiramate?",
    "Does topiramate affect oral contraceptives?",
    "What cognitive side effects are associated with topiramate?",

    # Valproate
    "What are the black box warnings for valproate?",
    "Is valproate safe during pregnancy?",
    "What is the mechanism of action of valproate?",
    "What liver toxicity risks are associated with valproate?",
    "How does valproate interact with lamotrigine?",

    # Carbamazepine
    "What are the contraindications for carbamazepine?",
    "What is the mechanism of action of carbamazepine?",
    "What serious hematologic reactions are associated with carbamazepine?",
    "Does carbamazepine interact with oral contraceptives?",
    "What HLA testing is recommended before starting carbamazepine?",

    # Cross-drug comparison
    "How does the mechanism of action compare between levetiracetam and brivaracetam?",
    "Which drug has a higher affinity for SV2A — levetiracetam or brivaracetam?",
    "How do the pregnancy risks compare between valproate and levetiracetam?",
    "Which AEDs require dose adjustment for renal impairment?",
    "How does the suicidal ideation warning compare between levetiracetam and brivaracetam?",
]

# Run with metrics
results_log = []

print(f"Running {len(questions)} questions...\n")
print("=" * 80)

total_start = time.time()

for i, q in enumerate(questions):
    start = time.time()

    retrieval_start = time.time()
    retrieved = retrieve(q, n_results=5)
    retrieval_time = time.time() - retrieval_start

    generation_start = time.time()
    answer = ask(q)
    generation_time = time.time() - generation_start

    total_time = time.time() - start

    distances  = retrieved["distances"][0]
    scores     = [round(1 - d, 3) for d in distances]
    avg_score  = round(sum(scores) / len(scores), 3)
    top_score  = round(max(scores), 3)
    drugs_hit  = list(set([m["drug"] for m in retrieved["metadatas"][0]]))

    results_log.append({
        "index"          : i + 1,
        "question"       : q,
        "answer"         : answer,
        "retrieval_time" : round(retrieval_time, 3),
        "generation_time": round(generation_time, 3),
        "total_time"     : round(total_time, 3),
        "avg_score"      : avg_score,
        "top_score"      : top_score,
        "drugs_hit"      : drugs_hit,
        "answer_length"  : len(answer.split()),
    })

    print(f"\n[{i+1}/{len(questions)}] Q: {q}")
    print(f"  Retrieval: {retrieval_time:.2f}s | Generation: {generation_time:.2f}s | Total: {total_time:.2f}s")
    print(f"  Top score: {top_score} | Avg score: {avg_score} | Drugs: {drugs_hit}")
    print(f"  Answer ({len(answer.split())} words):")
    print(f"  {answer[:300]}{'...' if len(answer) > 300 else ''}")

total_elapsed = time.time() - total_start
print("\n" + "=" * 80)
print(f"All {len(questions)} questions completed in {total_elapsed:.2f}s")

Running 50 questions...


[1/50] Q: What is the starting dose of levetiracetam for adults?
  Retrieval: 0.27s | Generation: 1.19s | Total: 1.46s
  Top score: 0.847 | Avg score: 0.794 | Drugs: ['levetiracetam']
  Answer (57 words):
  The starting dose of levetiracetam for adults is 1000 mg/day, given as twice-daily dosing (500 mg twice daily). This applies to monotherapy and adjunctive therapy for partial-onset seizures, as well as for primary generalized tonic-clonic seizures and myoclonic seizures in patients 12 years of age a...

[2/50] Q: What is the maximum daily dose of levetiracetam?
  Retrieval: 0.25s | Generation: 1.23s | Total: 1.48s
  Top score: 0.893 | Avg score: 0.822 | Drugs: ['levetiracetam']
  Answer (48 words):
  The maximum daily dose of levetiracetam is 3000 mg. This is consistent across different age groups and types of seizures, including partial-onset seizures, primary generalized tonic-clonic seizures, and myoclonic seizures in patients 12 years of age and older w

In [11]:
# Performance summary
retrieval_times  = [r["retrieval_time"]  for r in results_log]
generation_times = [r["generation_time"] for r in results_log]
total_times      = [r["total_time"]      for r in results_log]
avg_scores       = [r["avg_score"]       for r in results_log]
top_scores       = [r["top_score"]       for r in results_log]
answer_lengths   = [r["answer_length"]   for r in results_log]

# Categorise retrieval quality
excellent = sum(1 for s in top_scores if s >= 0.80)
good      = sum(1 for s in top_scores if 0.65 <= s < 0.80)
poor      = sum(1 for s in top_scores if s < 0.65)

# Drug coverage
all_drugs = [drug for r in results_log for drug in r["drugs_hit"]]
lev_count = all_drugs.count("levetiracetam")
bri_count = all_drugs.count("brivaracetam")
both      = sum(1 for r in results_log if len(r["drugs_hit"]) > 1)

print("\nPERFORMANCE METRICS SUMMARY")
print("=" * 55)

print("\nLatency")
print(f"  Retrieval  — avg: {statistics.mean(retrieval_times):.3f}s | min: {min(retrieval_times):.3f}s | max: {max(retrieval_times):.3f}s | stdev: {statistics.stdev(retrieval_times):.3f}s")
print(f"  Generation — avg: {statistics.mean(generation_times):.3f}s | min: {min(generation_times):.3f}s | max: {max(generation_times):.3f}s | stdev: {statistics.stdev(generation_times):.3f}s")
print(f"  End-to-end — avg: {statistics.mean(total_times):.3f}s | min: {min(total_times):.3f}s | max: {max(total_times):.3f}s | stdev: {statistics.stdev(total_times):.3f}s")

print("\nRetrieval Quality")
print(f"  Avg top-1 score     : {statistics.mean(top_scores):.3f}")
print(f"  Median top-1 score  : {statistics.median(top_scores):.3f}")
print(f"  Avg relevance score : {statistics.mean(avg_scores):.3f}")
print(f"  Best retrieval      : {max(top_scores):.3f} — Q{top_scores.index(max(top_scores))+1}: '{results_log[top_scores.index(max(top_scores))]['question'][:60]}'")
print(f"  Worst retrieval     : {min(top_scores):.3f} — Q{top_scores.index(min(top_scores))+1}: '{results_log[top_scores.index(min(top_scores))]['question'][:60]}'")
print(f"\n  Score distribution:")
print(f"    Excellent (>=0.80) : {excellent}/{len(questions)} questions ({excellent/len(questions)*100:.0f}%)")
print(f"    Good      (0.65-0.80): {good}/{len(questions)} questions ({good/len(questions)*100:.0f}%)")
print(f"    Poor      (<0.65)  : {poor}/{len(questions)} questions ({poor/len(questions)*100:.0f}%)")

print("\nAnswer Quality")
print(f"  Avg answer length   : {statistics.mean(answer_lengths):.0f} words")
print(f"  Median answer length: {statistics.median(answer_lengths):.0f} words")
print(f"  Shortest answer     : {min(answer_lengths)} words — Q{answer_lengths.index(min(answer_lengths))+1}")
print(f"  Longest answer      : {max(answer_lengths)} words — Q{answer_lengths.index(max(answer_lengths))+1}")

print("\nCoverage")
print(f"  Levetiracetam chunks retrieved : {lev_count} times")
print(f"  Brivaracetam chunks retrieved  : {bri_count} times")
print(f"  Questions hitting both drugs   : {both}/{len(questions)} ({both/len(questions)*100:.0f}%)")

print("\nCost Estimate")
input_tokens  = len(questions) * 1500
output_tokens = sum(answer_lengths) * 1.3
input_cost    = (input_tokens  / 1_000_000) * 0.80
output_cost   = (output_tokens / 1_000_000) * 3.20
print(f"  Input tokens  : ~{input_tokens:,}")
print(f"  Output tokens : ~{int(output_tokens):,}")
print(f"  Total cost    : ~${input_cost + output_cost:.4f}")


PERFORMANCE METRICS SUMMARY

Latency
  Retrieval  — avg: 0.280s | min: 0.252s | max: 0.471s | stdev: 0.056s
  Generation — avg: 1.464s | min: 0.850s | max: 3.318s | stdev: 0.487s
  End-to-end — avg: 1.743s | min: 1.146s | max: 3.576s | stdev: 0.509s

Retrieval Quality
  Avg top-1 score     : 0.777
  Median top-1 score  : 0.807
  Avg relevance score : 0.683
  Best retrieval      : 0.925 — Q16: 'What are the most common adverse reactions to brivaracetam?'
  Worst retrieval     : 0.579 — Q36: 'What are the black box warnings for valproate?'

  Score distribution:
    Excellent (>=0.80) : 26/50 questions (52%)
    Good      (0.65-0.80): 19/50 questions (38%)
    Poor      (<0.65)  : 5/50 questions (10%)

Answer Quality
  Avg answer length   : 80 words
  Median answer length: 65 words
  Shortest answer     : 15 words — Q10
  Longest answer      : 271 words — Q26

Coverage
  Levetiracetam chunks retrieved : 15 times
  Brivaracetam chunks retrieved  : 15 times
  Questions hitting both drugs 

In [12]:
# Per-question scorecard with assessment
print("\nPER-QUESTION SCORECARD")
print(f"{'#':<4} {'Score':<8} {'Grade':<10} {'Time':<8} {'Drugs':<25} {'Question'}")
print("-" * 95)

for r in results_log:
    score = r["top_score"]
    grade = "Excellent" if score >= 0.80 else "Good" if score >= 0.65 else "Poor"
    drugs = ", ".join(r["drugs_hit"])
    print(f"{r['index']:<4} {score:<8} {grade:<10} {r['total_time']:<8} {drugs:<25} {r['question'][:45]}")

print("\nQuestions flagged for review (score < 0.65):")
flagged = [r for r in results_log if r["top_score"] < 0.65]
if flagged:
    for r in flagged:
        print(f"  Q{r['index']}: {r['question']}")
        print(f"    Score: {r['top_score']} | Answer: {r['answer'][:150]}...")
else:
    print("  None — all questions above threshold.")


PER-QUESTION SCORECARD
#    Score    Grade      Time     Drugs                     Question
-----------------------------------------------------------------------------------------------
1    0.847    Excellent  1.462    levetiracetam             What is the starting dose of levetiracetam fo
2    0.893    Excellent  1.481    levetiracetam             What is the maximum daily dose of levetiracet
3    0.806    Excellent  1.66     levetiracetam             What is the recommended dose of levetiracetam
4    0.847    Excellent  1.324    levetiracetam             How should levetiracetam be dosed in patients
5    0.905    Excellent  1.669    levetiracetam             What is the dose adjustment for levetiracetam
6    0.713    Good       2.403    levetiracetam             Can levetiracetam cause aggression in childre
7    0.846    Excellent  1.323    levetiracetam             What dermatological reactions have been repor
8    0.733    Good       1.976    levetiracetam             What is t

## Check how good is our LLM

In [13]:
# Section 13 — Wrap Bedrock models for RAGAS
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_aws import ChatBedrock, BedrockEmbeddings
from datasets import Dataset

# Reuse the same region and credentials already set earlier in the notebook
llm = LangchainLLMWrapper(ChatBedrock(
    model_id="amazon.nova-pro-v1:0",
    client= client,
    model_kwargs={"temperature": 0}
))

embeddings = LangchainEmbeddingsWrapper(BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v2:0",
    client=embed_client
))

print("RAGAS LLM and embeddings ready.")

d:\miniconda3\envs\chainlit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Dwayne Reinaldy\AppData\Local\Temp\ipykernel_29596\4153608164.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\Dwayne Reinaldy\AppData\Local\Temp\ipykernel_29596\4153608164.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, con

RAGAS LLM and embeddings ready.


C:\Users\Dwayne Reinaldy\AppData\Local\Temp\ipykernel_29596\4153608164.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm = LangchainLLMWrapper(ChatBedrock(
C:\Users\Dwayne Reinaldy\AppData\Local\Temp\ipykernel_29596\4153608164.py:16: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(BedrockEmbeddings(


In [14]:
# Section 14 — Define evaluation questions and ground truths
eval_questions = [
    "What is the starting dose of levetiracetam for adults?",
    "Can brivaracetam be given intravenously?",
    "What is the maximum daily dose of brivaracetam for adults?",
    "How does renal impairment affect levetiracetam clearance?",
    "Is brivaracetam a controlled substance?",
    "What dermatological reactions have been reported for levetiracetam?",
    "What is the mechanism of action of brivaracetam?",
    "Is levetiracetam safe during pregnancy?",
    "Does topiramate affect oral contraceptives?",
    "What are the most common adverse reactions to brivaracetam?",
]

ground_truths = [
    "The starting dose of levetiracetam for adults is 1000 mg/day given as 500 mg twice daily.",
    "Yes, brivaracetam can be given intravenously at the same dosage and frequency as oral, administered over 2 to 15 minutes.",
    "The maximum daily dose of brivaracetam for adults is 200 mg per day administered as 100 mg twice daily.",
    "Levetiracetam clearance is reduced by 40% in mild, 50% in moderate, and 60% in severe renal impairment, correlated with creatinine clearance.",
    "Yes, brivaracetam is a Schedule V controlled substance.",
    "Stevens-Johnson syndrome and toxic epidermal necrolysis have been reported with levetiracetam, with median onset of 14 to 17 days.",
    "Brivaracetam displays high and selective affinity for synaptic vesicle protein 2A (SV2A) in the brain, which may contribute to its anticonvulsant effect.",
    "Levetiracetam is not contraindicated in pregnancy but requires monitoring as plasma levels may decrease during the third trimester.",
    "Topiramate may reduce the efficacy of oral contraceptives. Patients should use additional non-hormonal contraception.",
    "The most common adverse reactions to brivaracetam are somnolence and sedation (16%), dizziness (12%), fatigue (9%), and nausea/vomiting (5%).",
]

print(f"Evaluation set: {len(eval_questions)} questions")

Evaluation set: 10 questions


In [15]:
# Section 15 — Run RAG pipeline on evaluation questions
print("Running RAG pipeline on evaluation questions...")
answers  = []
contexts = []

for i, q in enumerate(eval_questions):
    retrieved = retrieve(q, n_results=5)
    answer    = ask(q)
    answers.append(answer)
    contexts.append(retrieved["documents"][0])
    print(f"  [{i+1}/{len(eval_questions)}] {q[:60]}")

print(f"\nAll {len(answers)} answers collected.")

Running RAG pipeline on evaluation questions...
  [1/10] What is the starting dose of levetiracetam for adults?
  [2/10] Can brivaracetam be given intravenously?
  [3/10] What is the maximum daily dose of brivaracetam for adults?
  [4/10] How does renal impairment affect levetiracetam clearance?
  [5/10] Is brivaracetam a controlled substance?
  [6/10] What dermatological reactions have been reported for levetir
  [7/10] What is the mechanism of action of brivaracetam?
  [8/10] Is levetiracetam safe during pregnancy?
  [9/10] Does topiramate affect oral contraceptives?
  [10/10] What are the most common adverse reactions to brivaracetam?

All 10 answers collected.


In [16]:
# Section 16 — Build RAGAS dataset and run evaluation
from datasets import Dataset

ragas_data = Dataset.from_dict({
    "question"    : eval_questions,
    "answer"      : answers,
    "contexts"    : contexts,
    "ground_truth": ground_truths,
})

print("Running RAGAS evaluation — takes 3 to 5 minutes...")

results = evaluate(
    dataset    = ragas_data,
    metrics    = [faithfulness, answer_relevancy, context_precision, context_recall],
    llm        = llm,
    embeddings = embeddings,
)

print("\nRAGAS RESULTS")
print("=" * 40)
print(results)

Running RAGAS evaluation — takes 3 to 5 minutes...


Evaluating: 100%|██████████| 40/40 [02:29<00:00,  3.73s/it]


RAGAS RESULTS
{'faithfulness': 1.0000, 'answer_relevancy': 0.8124, 'context_precision': 0.8956, 'context_recall': 1.0000}


In [19]:
import pandas as pd

df = results.to_pandas()

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.3f}".format)

# Run this first to see what columns RAGAS actually returned
print("Columns available:")
print(df.columns.tolist())
print()
print("Full results preview:")
print(df.head())

Columns available:
['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']

Full results preview:
                                                   user_input  \
0      What is the starting dose of levetiracetam for adults?   
1                    Can brivaracetam be given intravenously?   
2  What is the maximum daily dose of brivaracetam for adults?   
3   How does renal impairment affect levetiracetam clearance?   
4                     Is brivaracetam a controlled substance?   

                                            retrieved_contexts  \
0  [Drug: levetiracetam\nSection: Dosage and Administration...   
1  [Drug: brivaracetam\nSection: Dosage and Administration\...   
2  [Drug: brivaracetam\nSection: Dosage and Administration\...   
3  [Drug: levetiracetam\nSection: Use in Specific Populatio...   
4  [Drug: brivaracetam\nSection: DRUG ABUSE AND DEPENDENCE ...   

                                 

In [21]:
# Fixed Section 17 — correct column names
print("PER-QUESTION BREAKDOWN")
print("-" * 110)
print(df[["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]].to_string(index=False))

print("\nSUMMARY")
print("-" * 40)
print(f"  Faithfulness      : {df['faithfulness'].mean():.3f}  (target >0.95)")
print(f"  Answer Relevancy  : {df['answer_relevancy'].mean():.3f}  (target >0.80)")
print(f"  Context Precision : {df['context_precision'].mean():.3f}  (target >0.70)")
print(f"  Context Recall    : {df['context_recall'].mean():.3f}  (target >0.70)")

print("\nFlagged questions (faithfulness < 0.80):")
flagged = df[df["faithfulness"] < 0.80]
if len(flagged) > 0:
    for _, row in flagged.iterrows():
        print(f"  - {row['user_input'][:70]} | score: {row['faithfulness']:.3f}")
else:
    print("  None — all questions passed.")

PER-QUESTION BREAKDOWN
--------------------------------------------------------------------------------------------------------------
                                                         user_input  faithfulness  answer_relevancy  context_precision  context_recall
             What is the starting dose of levetiracetam for adults?         1.000             0.799              1.000           1.000
                           Can brivaracetam be given intravenously?         1.000             0.948              1.000           1.000
         What is the maximum daily dose of brivaracetam for adults?         1.000             0.959              0.867           1.000
          How does renal impairment affect levetiracetam clearance?         1.000             0.881              1.000           1.000
                            Is brivaracetam a controlled substance?         1.000             0.905              1.000           1.000
What dermatological reactions have been reported for lev